# 7 - Profiles, Scenarios, and the Two Consumers

A **profile** pairs a base selection with a factor selection and a scenario mode. Resolving a profile expands families/groups and filters factors by `applies_to`. Composing turns the resolved suite + base items into concrete **scenarios**, which feed either consumer: SFT data generation (Ch15) or agent testing (Ch16).

In [ ]:
import sys, pathlib
import proofloop.suite as gt
cat = gt.Catalog.load()
print(cat.summary())

## Profiles ship as pick-and-choose bundles

In [ ]:
for name, p in cat.profiles.items():
    print(f'{name:20s} mode={p.mode:9s} bases={p.bases} factors={p.factors[:3]}...')

In [ ]:
suite = gt.resolve_profile(cat, 'numeric_screen')
print(suite.summary())

## Two scenario modes
**cross** (Option 1): base x design matrix - every base seen under every design row, size = bases * n_runs.

**embedded** (Option 2): the base is a *balanced blocking* factor - every base exercised equally, size = n_runs. (At small n this avoids the joint phi_p design starving a high-cardinality base; see the JASA paper.)

In [ ]:
from proofloop.suite.compose import graphdoe_design   # real knowlytix Sobol+refine designs
from collections import Counter
items = [gt.QAItem(qid=f'q{i}', query=f'metric {i}?', answer=str(i)) for i in range(5)]
cross = gt.compose(gt.resolve(cat, ['exact_recall'], ['clarity'], mode='cross'),
                   items, n_runs=8, seed=1, design_fn=graphdoe_design)
emb   = gt.compose(gt.resolve(cat, ['exact_recall'], ['clarity'], mode='embedded'),
                   items, n_runs=8, seed=1, design_fn=graphdoe_design)
print('cross   ', len(cross), 'scenarios')
print('embedded', len(emb), 'scenarios; base balance', dict(Counter(s.base.qid for s in emb)))

## The user-supplied variant (no GMS)
Bring your own `(query, answer)` set, enrich it with presentation factors, and you never touch a store. Ground truth is your supplied answer; the queries are varied for coverage, not expanded by a graph.

In [ ]:
user_items = gt.UserBaseSource([
    {'query': 'Was I overcharged on my overdraft?', 'answer': 'complaint'},
    {'query': 'How do I close my account?',         'answer': 'inquiry'},
]).items()
u_suite = gt.resolve(cat, ['exact_recall'], ['clarity','noise'], mode='cross')
u_scn = gt.compose(u_suite, user_items, n_runs=4, seed=2)
print(len(u_scn), 'scenarios from user pairs; ground truth preserved:',
      all(s.base.answer in ('complaint','inquiry') for s in u_scn))

## Consumer A - SFT data (Chapter 15)
`emit_classifier_sft` produces label-preserving `(message, label)` records; `emit_draft_sft` produces grounded `(user, assistant)` pairs validated against a golden contract.

In [ ]:
recs = gt.emit_classifier_sft(u_scn)
print('records:', len(recs), '| keys:', sorted(recs[0]))
print('labels preserved:', {r['label'] for r in recs})

## Consumer B - agent testing (Chapter 16)
`evaluate.run` sends each scenario through a System Under Test - any `(query, context) -> answer` callable - and scores outcome, trajectory and process. Here a trivial SUT stands in for the real agent (which `apps/complaint_sut.AgentSUT` wraps).

In [ ]:
from proofloop.suite.evaluate import run, summary
rows = run(u_scn, lambda q, context='': 'complaint' if 'overcharged' in q else 'inquiry')
print(summary(rows))

## Visualization: coverage of a space-filling design
The design generator favors Sobol+refine because it spreads points evenly. Random points clump and leave gaps; Sobol fills the space.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pathlib
FIGDIR = pathlib.Path("figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
import numpy as np
from scipy.stats import qmc
rand = np.random.default_rng(0).random((64, 2))
sob = qmc.Sobol(d=2, scramble=True, seed=0).random_base2(m=6)   # 64 points
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
for ax, (pts, title) in zip(axes, [(rand, 'random'), (sob, 'Sobol (space-filling)')]):
    ax.scatter(pts[:, 0], pts[:, 1], s=14, color='#36c')
    ax.set_title(title); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
fig.suptitle('64 points over two factors')
fig.tight_layout(); fig.savefig(FIGDIR / 'fig_design_coverage.png', dpi=150)
print('wrote', FIGDIR / 'fig_design_coverage.png')

## Self-check

In [ ]:
assert len(cross) == 5 * 8 and len(emb) == 8        # cross = bases*runs, embedded = runs
assert set(recs[0]) == {'message', 'label', '_seed', '_factors'}
assert all(s.base.answer in ('complaint', 'inquiry') for s in u_scn)  # GT preserved, no GMS
print('OK - profiles & scenarios')